In [ ]:
import subprocess, glob, os, sys, types, warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

fake_mamba3 = types.ModuleType('mamba_ssm.modules.mamba3')
fake_mamba3.Mamba3 = type('Mamba3', (), {})
sys.modules['mamba_ssm.modules.mamba3'] = fake_mamba3

import mamba_ssm; print(f'mamba_ssm {mamba_ssm.__version__} OK')

In [ ]:
pkg=glob.glob('/kaggle/input/**/packages/',recursive=True)
if pkg: subprocess.run(f'pip install -q --no-index --find-links {pkg[0]} trl --no-deps',shell=True)
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
import trl, torch, torch.nn.functional as F, kagglehub, stat, shutil, zipfile, re, pandas as pd, numpy as np
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
print(f'trl {trl.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)} | {torch.cuda.get_device_properties(0).total_memory/1e9:.0f}GB')
LORA_RANK=32; LORA_ALPHA=32; MAX_SEQ_LEN=2048; NUM_EPOCHS=1
GRAD_ACCUM=8; LR=2e-4; WARMUP_STEPS=0; MAX_STEPS=60
OUTPUT_DIR='/kaggle/working/sft_output'; os.makedirs(OUTPUT_DIR,exist_ok=True)
ADAPTER_DIR='/kaggle/working/sft_adapter'; os.makedirs(ADAPTER_DIR,exist_ok=True)
DATA_DIR=os.path.dirname(glob.glob('/kaggle/input/**/train.csv',recursive=True)[0])
print('DATA_DIR:', DATA_DIR)

In [ ]:
src='/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell'
if os.path.exists(src):
    dst='/tmp/ptxas-blackwell'
    shutil.copy2(src,dst); os.chmod(dst,os.stat(dst).st_mode|stat.S_IEXEC|stat.S_IXGRP|stat.S_IXOTH)
def _rmsnorm(x,weight,bias=None,z=None,eps=1e-5,**kw):
    dtype=x.dtype; x=x.float()
    out=x*torch.rsqrt(x.pow(2).mean(-1,keepdim=True)+eps)*weight.float()
    if bias is not None: out+=bias.float()
    if z is not None: out*=F.silu(z.float())
    return out.to(dtype)
for _,m in list(sys.modules.items()):
    if hasattr(m,'rmsnorm_fn'): m.rmsnorm_fn=_rmsnorm
print('Triton OK')

In [ ]:
train_df=pd.read_csv(f'{DATA_DIR}/train.csv'); print(f'Base train: {len(train_df)}')
MODEL_PATH=kagglehub.model_download('metric/nemotron-3-nano-30b-a3b-bf16/transformers/default')
tokenizer=AutoTokenizer.from_pretrained(MODEL_PATH,trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
PROMPT_SUFFIX='\nPut your final answer inside \\boxed{}.'

# CoT dataset (dgxchen/nemotron-cot-tong)
cot_paths=glob.glob('/kaggle/input/**/problem_ids_matched.csv',recursive=True)
if cot_paths:
    cot_df=pd.read_csv(cot_paths[0])
    print(f'CoT dataset: {len(cot_df)} rows, cols: {list(cot_df.columns)}')
    records=[]
    for _,row in cot_df.iterrows():
        prompt=str(row['prompt']); answer=str(row['answer'])
        cot=str(row.get('generated_cot',''))
        if not cot or cot=='nan' or len(cot.strip())<5: continue
        cot_cleaned=re.sub(r'\\boxed\{[^}]*\}','',cot).rstrip()
        user_content=prompt+PROMPT_SUFFIX
        assistant_content=f'<think>\n{cot_cleaned}\n</think>\n\\boxed{{{answer}}}'
        try:
            text=tokenizer.apply_chat_template(
                [{'role':'user','content':user_content},{'role':'assistant','content':assistant_content}],
                tokenize=False,add_generation_prompt=False)
        except:
            text=f'<|im_start|>user\n{user_content}<|im_end|>\n<|im_start|>assistant\n{assistant_content}<|im_end|>'
        records.append({'text':text})
    dataset=Dataset.from_list(records)
    print(f'CoT records: {len(dataset)}')
else:
    print('No CoT dataset -- falling back to plain SFT on train.csv')
    def build_text(ex):
        u=ex['prompt']+PROMPT_SUFFIX; a='\\boxed{'+ex['answer']+'}'
        try: return {'text':tokenizer.apply_chat_template([{'role':'user','content':u},{'role':'assistant','content':a}],tokenize=False,add_generation_prompt=False)}
        except: return {'text':'<|im_start|>user\n'+u+'<|im_end|>\n<|im_start|>assistant\n'+a+'<|im_end|>'}
    dataset=Dataset.from_pandas(train_df).map(build_text,remove_columns=['id','prompt','answer'])
    print(f'Dataset: {len(dataset)}')

lengths=[len(tokenizer.encode(t)) for t in dataset['text'][:50]]
print(f'Token lengths (first 50): min={min(lengths)} max={max(lengths)} mean={np.mean(lengths):.0f} median={np.median(lengths):.0f}')

In [ ]:
print('Loading model...')
model=AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,device_map='auto',trust_remote_code=True,dtype=torch.bfloat16,
    attn_implementation='eager')
model.config.use_cache=False
model.config.pad_token_id=tokenizer.pad_token_id
model.config.eos_token_id=tokenizer.eos_token_id
for name,mod in sys.modules.items():
    if 'modeling_nemotron_h' in name and hasattr(mod,'is_fast_path_available'):
        mod.is_fast_path_available=False; print(f'Patched {name}')
print(f'Params: {sum(p.numel() for p in model.parameters())/1e9:.1f}B')

In [ ]:
model=get_peft_model(model,LoraConfig(
    r=LORA_RANK,lora_alpha=LORA_ALPHA,
    target_modules=['q_proj','k_proj','v_proj','o_proj','in_proj','out_proj','up_proj','down_proj'],
    lora_dropout=0.0,bias='none',task_type=TaskType.CAUSAL_LM))
model.print_trainable_parameters()

In [ ]:
os.environ['TRITON_PTXAS_BLACKWELL_PATH']='/tmp/ptxas-blackwell'
os.environ['TQDM_DISABLE']='1'
try:
    import triton.backends.nvidia.compiler as nvc; nvc.get_ptxas_version=lambda arch:'12.0'
except Exception as e: print(f'Triton compiler: {e}')

class PrintProgress(TrainerCallback):
    def on_log(self,args,state,control,logs=None,**kwargs):
        if logs and state.global_step and 'loss' in logs:
            print(f"[step {state.global_step}/{state.max_steps}] "
                  f"loss={logs['loss']:.4f}  "
                  f"lr={logs.get('learning_rate',0):.2e}  "
                  f"epoch={state.epoch:.2f}",flush=True)

trainer=SFTTrainer(model=model,train_dataset=dataset,processing_class=tokenizer,
    callbacks=[PrintProgress()],
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=NUM_EPOCHS,
        max_steps=MAX_STEPS,
        learning_rate=LR,
        lr_scheduler_type='linear',
        warmup_steps=WARMUP_STEPS,
        adam_beta2=0.95,
        logging_steps=5,
        bf16=True,
        max_grad_norm=1.0,
        optim='adamw_torch',
        save_strategy='no',
        report_to='none',
        dataset_text_field='text',
        max_length=MAX_SEQ_LEN,
        packing=False,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant':False},
        dataloader_num_workers=2,
        remove_unused_columns=False,
        seed=42,
    ))
print(f'Training... packing=False, max_length={MAX_SEQ_LEN}, max_steps={MAX_STEPS}, grad_accum={GRAD_ACCUM}')
trainer.train()
print('Done.')

In [ ]:
import json
model.save_pretrained(ADAPTER_DIR)
with open(f'{ADAPTER_DIR}/adapter_config.json') as f: cfg=json.load(f)
cfg['base_model_name_or_path']='nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'
cfg['inference_mode']=True
cfg['lora_dropout']=0.0
with open(f'{ADAPTER_DIR}/adapter_config.json','w') as f: json.dump(cfg,f,indent=2)

zip_path='/kaggle/working/submission.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
    [zf.write(os.path.join(ADAPTER_DIR,fname),fname) for fname in os.listdir(ADAPTER_DIR)
     if os.path.isfile(os.path.join(ADAPTER_DIR,fname))]
with zipfile.ZipFile(zip_path) as zf: names=zf.namelist()
assert 'adapter_config.json' in names
print(f'Done. {os.path.getsize(zip_path)/1024/1024:.1f}MB: {names}')